<style>

.footnote-list {
    display: none;
}
</style>

# Basic BPE Tokenizer
A trained BPE takes an input text, splits it up into tokens, and assigns a token ID (not to be confused with the numerical vector embeddings of said token):

```{figure} ../../figures/class1/004_BPE.png
---
name: BPE-high-level
---
Modified from [Sebastian Rasckha](https://github.com/rasbt/LLMs-from-scratch/blob/main) under the [APACHE License](https://github.com/rasbt/LLMs-from-scratch/blob/main/LICENSE.txt).

This is what we'll aim to do!

## 4.1 Training
We begin the process with an initial vocabulary that is a set of all individual characters[^special_ex]

Then we do these these steps:
1. **Count** the most frequent combinations of **characters** *or* **bytes**[^bytes_ex] in our corpus 
2. **Merge** that pair 
3. **Replace** that pair with a token ID
3. **Repeat** 1 & 2 until there are no gains 
    - Or until we've hit "max" vocabulary length, set by a parameter *vocab_size* or *k* in [Jurafsky & Martin](https://web.stanford.edu/~jurafsky/slp3/2.pdf).

[^special_ex]: Some implementations add "special character tokens". We won't do this here. See [Wiki/BPE](https://en.wikipedia.org/wiki/Byte-pair_encoding#Modified_algorithm)
[^bytes_ex]: LLMs use Byte-level BPE, not character-level. We'll start with words for intution, then transition to bytes.

### Step 1: Finding Frequent Combinations
Let's start with an example of a simple sequence:


In [84]:
input_text = "the cat sat on the mat"

In Python, we can easily iterate over characters in a string. Let's do that with `enumerate` where we can also get the position of the character: 

In [86]:
for i, char in enumerate(input_text):
    print(i, char)

0 t
1 h
2 e
3  
4 c
5 a
6 t
7  
8 s
9 a
10 t
11  
12 o
13 n
14  
15 t
16 h
17 e
18  
19 m
20 a
21 t


#### Construct Pairs
We need to construct pairs. That is, go through each `char` and combine with `next_char`. By hand, this for loop looks like this:
```
iteration 1: 
    char = t 
    next_char = h
    pair = (char, next_char)
iteration 2: 
    char = h 
    next_char = e
    pair = (char, next_char)
```

:::{admonition} HANDS-ON: Identify the `next_char` & create pairs!
:class: red
Loop over characters with `enumerate`: 
1. Identify `char` and `next_char`
2. Save these as a pair 
3. Print the pair
4. Remember to end the for loop *before* you check the last character, since that one won't have a consecutive pair to check with!
:::: 

##### Solution

:::{admonition} How to identify `next_char`
:class: tip, dropdown
Each character in a string has a position. We can select each character with this position e.g., text[2] would be "e"[^zero_index]. With `enumerate`, we've made that position explicit using `i, char`. `i` is the current char, how would we get the next one?

<details>
<summary>Click to see answer</summary>
We identify the <code>next_char</code> by adding +1 to <code>i</code>, i.e., <code>text[i + 1]</code>
</details>
::::

[^zero_index]: Remember that Python is zero-indexed!

:::{admonition} End the loop before the last character.
:class: tip, dropdown
When we get to the last character, you need to ensure that 
::::

In [87]:
for i, char in enumerate(input_text[:-1]):
    next_char = input_text[i + 1]
    pair = (char, next_char)
    print(pair)

('t', 'h')
('h', 'e')
('e', ' ')
(' ', 'c')
('c', 'a')
('a', 't')
('t', ' ')
(' ', 's')
('s', 'a')
('a', 't')
('t', ' ')
(' ', 'o')
('o', 'n')
('n', ' ')
(' ', 't')
('t', 'h')
('h', 'e')
('e', ' ')
(' ', 'm')
('m', 'a')
('a', 't')


#### Count Combinations
Now we need to count the pairs and add their counts to a dictionary, with each pair being a `key` and their count being a `value`. This is the end goal:
```
counts = {('t', 'h'): 2, ('h', 'e'): 2, ('e', ' '): 2, (' ', 'c'): 1, ('c', 'a'): 1, ('a', 't'): 3, ...}
```

Below is a starting point of a function that counts the combinations !

In [ ]:
def count_combinations(input):
    # initialize a counts dictionary which is empty if there aren't any combinations yet!
    counts = {} if counts is None else counts 

    # your code

    return counts

#### Encode Text!

The example above used *characters*. No matter which language or symbol, the **Unicode** character standard has a number id for this character (also called **Code Points**). In other words, it is our giant "master list", containing more than 150000 code points! You can explore them [here](https://unicode-explorer.com/).

For instance, the Danish Å, Ø, Æ are represented in UNICODE as:
```python
Å = U+00C5
Ø = U+00D8
Æ = U+00C6
```

We could intialize our BPE with a vocabulary of +150000 unicode code points to cover every possible character. But that would be pretty inefficient. Instead, we can encode the characters as **sequences** of bytes using UTF-8 encoding. 

For the Danish Å, Ø, Æ, UTF-8 gives us these 2-byte sequences:
```python
Å = [195, 133]
Ø = [195, 152]
Æ = [195, 134]
```

A byte can have 256 possible values, from 0 to 255. 

Note that this representation is in decimal numbers[^ord_ex], but that Jurafsky & Martin explain it with hexidecimals: 
```python
decimal      hexadecimal      unicode (i.e., with prefix)

104          68               U+0068
101          65               U+0065
108          6C               U+006C
108          6C               U+006C
111          6F               U+006F
```
[^ord_ex]: We can use the function `ord()` in Python to convert characters into decimal numbers!

We can literally just encode our text, and then use our counting approach. It'll work the same

This does not tell us much! We need a way to look at them pairwise:

In [ ]:
#tokens = list(text.encode("utf-8"))

def get_stats(ids, counts=None):
    """
    Given a list of integers, return a dictionary of counts of consecutive pairs
    Example: [1, 2, 3, 1, 2] -> {(1, 2): 2, (2, 3): 1, (3, 1): 1}
    Optionally allows to update an existing dictionary of counts
    """
    counts = {} if counts is None else counts
    for pair in zip(ids, ids[1:]): # iterate consecutive elements
        counts[pair] = counts.get(pair, 0) + 1
    return counts



In [75]:
stats = get_stats(text)
print(f"Total number of unique pairs: {len(stats)}")
print(stats)

# Show top 10 most frequent pairs
top_pairs = sorted([(count, pair) for pair, count in stats.items()], reverse=True)[:10]
print("\nTop 10 most frequent pairs:")
for count, pair in top_pairs:
    print(f"  {pair}: {count} times")

print(top_pairs)

Total number of unique pairs: 15
{('t', 'h'): 2, ('h', 'e'): 2, ('e', ' '): 2, (' ', 'c'): 1, ('c', 'a'): 1, ('a', 't'): 3, ('t', ' '): 2, (' ', 's'): 1, ('s', 'a'): 1, (' ', 'o'): 1, ('o', 'n'): 1, ('n', ' '): 1, (' ', 't'): 1, (' ', 'm'): 1, ('m', 'a'): 1}

Top 10 most frequent pairs:
  ('a', 't'): 3 times
  ('t', 'h'): 2 times
  ('t', ' '): 2 times
  ('h', 'e'): 2 times
  ('e', ' '): 2 times
  ('s', 'a'): 1 times
  ('o', 'n'): 1 times
  ('n', ' '): 1 times
  ('m', 'a'): 1 times
  ('c', 'a'): 1 times
[(3, ('a', 't')), (2, ('t', 'h')), (2, ('t', ' ')), (2, ('h', 'e')), (2, ('e', ' ')), (1, ('s', 'a')), (1, ('o', 'n')), (1, ('n', ' ')), (1, ('m', 'a')), (1, ('c', 'a'))]


### 1. Count **Encoded** Pairs: Unicode, Bits & Bytes
The example above used *characters*, but in reality we're using Unicode, a **character set** where all characters have number id (**Code Points**). For Hello this would be:

```
U+0068 U+0065 U+006C U+006C U+006F
```
Unicode has +1500000 **code points**. This is not computionally efficient. Luckily, we can convert this into bytes with the **UTF-8 encoding**:

In [29]:
text = "hello"
encoding = list(text.encode("utf-8"))
print(encoding)

[104, 101, 108, 108, 111]


Note that this is represented in decimal numbers[^ord_ex], but that Jurafsky & Martin explain it with hexidecimals: 
```python
decimal    hexadecimal
104        68
101        65
108        6C
108        6C
111        6F
```

[^ord_ex]: We can use the function `ord()` in Python to convert characters into decimal numbers!

In [30]:
def get_stats(ids, counts=None):
    """
    Given a list of integers, return a dictionary of counts of consecutive pairs
    Example: [1, 2, 3, 1, 2] -> {(1, 2): 2, (2, 3): 1, (3, 1): 1}
    Optionally allows to update an existing dictionary of counts
    """
    counts = {} if counts is None else counts
    for pair in zip(ids, ids[1:]): # iterate consecutive elements
        counts[pair] = counts.get(pair, 0) + 1
    return counts

## References
This NB is partially inspired by Sebastian Rasckha's "Build a Large Language Model from Scratch" and HF's [BPE tutorial](https://huggingface.co/learn/llm-course/en/chapter6/5), both licensed under APACHE.

``